# VAYU Climate Digital Twin â€” Kaggle GPU Training

**Accelerator**: GPU T4 Ã— 1 (16 GB) or P100 (16 GB)  
**Target**: Train VayuClimateModel on IMD 2010-2024, validate 2021-2023, test 2024  
**Dataset**: Upload `data/processed/` directory as Kaggle Dataset named `vayu-imd-processed`

## Setup
1. Enable GPU: Settings â†’ Accelerator â†’ GPU T4 x1
2. Add Dataset: `vayu-imd-processed` (your uploaded processed NetCDF files)
3. Run all cells top to bottom

In [ ]:
# â”€â”€ Environment check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected â€” switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
# â”€â”€ Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# torch-geometric version must match torch; use 2.5.3 for torch 2.x on Kaggle.
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
# ── Mount project code and locate dataset ─────────────────────────────────────
import sys, os
from pathlib import Path

REPO_DIR = '/kaggle/working/isro'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/wg_main'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Pull latest code if repo already exists, else clone fresh
if os.path.exists(REPO_DIR):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# Locate the uploaded Kaggle dataset (vayu-western-ghats-processed-v1)
root = Path('/kaggle/input')
required = [
    'rainfall_2010-2025.nc', 'tmax_2010-2025.nc', 'tmin_2010-2025.nc',
    'normalized_2010-2025.nc', 'pipeline_log_2010-2025.json',
]
found = {name: next(iter(root.rglob(name)), None) for name in required}
missing = [k for k, v in found.items() if v is None]
if missing:
    raise RuntimeError('Missing dataset files: ' + ', '.join(missing) +
                       ". Attach the dataset via 'Add Input'.")
parent_counts = {}
for p in found.values():
    parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
DATASET_DIR = max(parent_counts, key=parent_counts.get)
print('Dataset dir:', DATASET_DIR)


In [ ]:
# -- Copy raw files and run preprocessing -----------------------------------
import os, subprocess, sys

PY = sys.executable  # use the same Python the notebook kernel runs

os.makedirs(f'{REPO_DIR}/data/imd', exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/processed_western_ghats', exist_ok=True)
for f in ['rainfall_2010-2025.nc','tmax_2010-2025.nc','tmin_2010-2025.nc']:
    os.system(f'cp "{DATASET_DIR}/{f}" {REPO_DIR}/data/imd/')
for f in ['normalized_2010-2025.nc','pipeline_log_2010-2025.json']:
    os.system(f'cp "{DATASET_DIR}/{f}" {REPO_DIR}/data/processed_western_ghats/')

subprocess.run([PY,'-m','data_ingestion.cli','preprocess',
    '--data-dir', f'{REPO_DIR}/data/imd',
    '--output-dir', f'{REPO_DIR}/data/processed_western_ghats',
    '--start-year','2010','--end-year','2025',
    '--region','western_ghats','--resolution','0.25'],
    check=True, cwd=REPO_DIR)

# 1024 train / 256 val with stride-2 fully uses 15 years of data
subprocess.run([PY,'-m','data_ingestion.cli','build-sequences',
    '--normalized-file', f'{REPO_DIR}/data/processed_western_ghats/normalized_2010-2025.nc',
    '--output-dir', f'{REPO_DIR}/data/processed_western_ghats',
    '--input-window','30','--target-window','7',
    '--max-train','1024','--max-val','256',
    '--stride','2','--fillna-value','0.0'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {REPO_DIR}/data/processed_western_ghats')


In [ ]:
# -- Smoke check: verify model and data before full training ----------------
import subprocess, sys
PY = sys.executable

subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir', f'{REPO_DIR}/data/processed_western_ghats',
    '--checkpoint-dir', f'{REPO_DIR}/checkpoints/wg_smoke',
    '--epochs','1','--device','auto','--smoke-only'],
    check=True, cwd=REPO_DIR)


In [ ]:
# -- Full training: 2.3M params, physics ON, no quality compromise ----------
# Memory on T4 (15360 MiB): full model + AMP + bs=1 uses ~300 MB/step (50x margin)
# --amp              : FP16, halves activation memory
# --batch-size 1     : one sequence per step (the OOM fix; no preset needed)
# --grad-accum-steps 8 : effective batch=8 without extra VRAM
# No --kaggle-medium/lite: full 2.3M model + physics constraints ON
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir',    f'{REPO_DIR}/data/processed_western_ghats',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--epochs',      '60',
    '--device',      'auto',
    '--amp',
    '--batch-size',  '1',
    '--grad-accum-steps', '8',
    '--norm-params-file', f'{REPO_DIR}/data/processed_western_ghats/norm_params_2010-2025.nc',
    '--run-baselines',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')


In [ ]:
# â”€â”€ Load history and plot training curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet â€” run the training cell first.')
else:
    history = json.loads(history_path.read_text())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'],   label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

    axes[0].set_title('VAYU Training Loss'); axes[0].legend(); axes[0].grid(True)    print('Best val_loss:', min(history['val_loss']))

    plt.show()

    axes[1].plot(history['epochs'], history['val_r2'], label='Val RÂ² (Tmax)', color='green')    plt.savefig('/kaggle/working/training_curves.png', dpi=150)

    axes[1].axhline(0.85, color='red', linestyle='--', label='Target RÂ²=0.85')    plt.tight_layout()

    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RÂ²')
    axes[1].set_title('Validation RÂ² Score'); axes[1].legend(); axes[1].grid(True)

In [ ]:
# â”€â”€ Save best checkpoint for download â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    shutil.copy(best_ckpt, '/kaggle/working/vayu_best.pt')
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: /kaggle/working/vayu_best.pt ({size_mb:.1f} MB)')
    print('Download and upload to S3:')
    print('  aws s3 cp vayu_best.pt s3://vayu-models/checkpoints/')
else:
    print('vayu_best.pt not found â€” check training cell output for errors.')

## Next Steps After Training

1. **Download** `vayu_best.pt` from Kaggle Output
2. **Upload to S3**:
   ```bash
   aws s3 cp vayu_best.pt s3://vayu-climate-models/checkpoints/vayu_best.pt
   ```
3. **Trigger ECS deployment** (CDK will mount S3 checkpoint automatically)
4. **Verify**: `curl https://api.vayu-climate.com/health`

## Kaggle Quota Tips
- Each run â‰ˆ 2-4 hours on T4 (30h/week quota)
- Save checkpoint every 5 epochs to resume if quota runs out
- Use `early_stopping_patience=15` to auto-stop when converged
- Enable Accelerator **T4 x2** for 2Ã— speed if available